# Задание 1. Аналитика добычиВитрины: добыча по дням, KPI по скважинам, влияние T/P

In [ ]:
import os
import warnings

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from botocore.client import Config
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_URL = (
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'oiluser')}:"
    f"{os.getenv('POSTGRES_PASSWORD', 'oilpass')}@"
    f"{PG_HOST}:{os.getenv('POSTGRES_PORT', '5432')}/"
    f"{os.getenv('POSTGRES_DB', 'oildb')}"
)
engine = create_engine(PG_URL)

def minio_endpoint_url():
    raw = os.getenv("MINIO_ENDPOINT", "localhost:9000").strip().rstrip("/")
    if raw.startswith("http://") or raw.startswith("https://"):
        return raw
    return f"http://{raw}"

def minio_client():
    return boto3.client(
        "s3",
        endpoint_url=minio_endpoint_url(),
        aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "minioadmin"),
        aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "minioadmin"),
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )


In [ ]:

df = pd.read_sql("""
    SELECT p.date, w.well_id, w.name, p.oil_ton, p.pressure, p.temperature,
           p.downtime_hours, p.downtime_hours/24.0 AS downtime_ratio
    FROM production p JOIN wells w ON w.well_id = p.well_id
    WHERE p.oil_ton > 0
""", engine)

daily = df.groupby("date", as_index=False).agg(total_oil_ton=("oil_ton", "sum"))
print(daily)

kpi = df.groupby(["well_id", "name"], as_index=False).agg(
    avg_oil_ton=("oil_ton", "mean"),
    downtime_pct=("downtime_ratio", "mean"),
)
kpi["downtime_pct"] = (kpi["downtime_pct"] * 100).round(2)
print(kpi.sort_values("avg_oil_ton", ascending=False))

best = kpi.nlargest(2, "avg_oil_ton")
worst = kpi.nsmallest(2, "avg_oil_ton")
print("Лучшие:\n", best[["name", "avg_oil_ton"]])
print("Худшие:\n", worst[["name", "avg_oil_ton"]])

corr = df[["oil_ton", "temperature", "pressure"]].corr()
print("\nКорреляция с дебитом:\n", corr["oil_ton"])



In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(daily["date"], daily["total_oil_ton"], marker="o")
axes[0].set_title("Добыча по дням (Line)")
axes[0].tick_params(axis="x", rotation=45)

kpi.sort_values("avg_oil_ton").plot.barh(x="name", y="avg_oil_ton", ax=axes[1], legend=False)
axes[1].set_title("Топ скважин (Bar)")

pivot = df.pivot_table(values="oil_ton", index=pd.cut(df["pressure"], 5), columns=pd.cut(df["temperature"], 5), aggfunc="mean")
sns.heatmap(pivot, ax=axes[2], cmap="YlOrRd")
axes[2].set_title("Давление vs Дебит (Heatmap)")
plt.tight_layout()
plt.show()

